In [ ]:
# 06_comparison_of_DE_methods.ipynb
# here, we will compare Wilcoxon ranked list and pseudobulk results

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

wilcoxon_de = pd.read_csv("../data/interim/04_wilcoxon_de_results.csv.gz")
pseudobulk_de = pd.read_csv("../data/interim/05_pseudobulk_de_results.csv.gz")

print(wilcoxon_de.shape, wilcoxon_de.columns.tolist())
print(pseudobulk_de.shape, pseudobulk_de.columns.tolist())

In [ ]:
# Panel 1 data: top-hit overlap (Jaccard) per target, using genes shared by both methods
def jaccard_significant(target, alpha=0.05):
    w = set(wilcoxon_de[(wilcoxon_de["target"] == target) & (wilcoxon_de["pvals_adj"] < alpha)]["names"])
    p = set(pseudobulk_de[(pseudobulk_de["target"] == target) & (pseudobulk_de["padj"] < alpha)]["gene_name"])
    if not w or not p:
        return np.nan
    return len(w & p) / len(w | p)

jaccard_scores = pd.Series({t: jaccard_significant(t) for t in shared_targets}).dropna()
print(jaccard_scores.describe())

In [ ]:
# Panel 2 data: KLF1 effect-size concordance, gene-by-gene
w_klf1 = wilcoxon_de[wilcoxon_de["target"] == "KLF1"][["names", "logfoldchanges", "pvals_adj"]]
p_klf1 = pseudobulk_de[pseudobulk_de["target"] == "KLF1"][["gene_name", "log2FoldChange", "padj"]]
merged_klf1 = w_klf1.merge(p_klf1, left_on="names", right_on="gene_name")
print(merged_klf1.shape)

In [ ]:
# Panel 3 data: pseudobulk LFC vs baseMean, across ALL targets -- shows whether the
# low-count/high-LFC noise pattern we saw in KLF1 is a general phenomenon
pb_all = pseudobulk_de.copy()
pb_all["significant"] = pb_all["padj"] < 0.05

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# Panel A: distribution of top-20 Jaccard overlap scores across all shared targets
ax0 = axes[0]
ax0.hist(jaccard_scores, bins=20, color="#4C72B0", edgecolor="white", alpha=0.85)
ax0.axvline(jaccard_scores.median(), color="black", linewidth=1,
            label=f"median = {jaccard_scores.median():.2f}")
ax0.set_xlabel("Jaccard overlap (padj < 0.05 gene sets, Wilcoxon vs. pseudobulk)")
ax0.set_ylabel("number of targets")
ax0.set_title("A. Significant-gene agreement\nbetween methods")
ax0.legend()

# Panel B: KLF1 effect size concordance, gene-by-gene
ax1 = axes[1]
both_sig = (merged_klf1["pvals_adj"] < 0.05) & (merged_klf1["padj"] < 0.05)
ax1.scatter(merged_klf1.loc[~both_sig, "logfoldchanges"], merged_klf1.loc[~both_sig, "log2FoldChange"],
            s=8, alpha=0.3, color="lightgrey", label="not significant in both")
ax1.scatter(merged_klf1.loc[both_sig, "logfoldchanges"], merged_klf1.loc[both_sig, "log2FoldChange"],
            s=8, alpha=0.5, color="#C44E52", label="significant in both (padj<0.05)")
lims = [merged_klf1[["logfoldchanges", "log2FoldChange"]].min().min(),
        merged_klf1[["logfoldchanges", "log2FoldChange"]].max().max()]
ax1.plot(lims, lims, color="black", linestyle="--", linewidth=1, label="y = x")
ax1.set_xlabel("Wilcoxon log2FC")
ax1.set_ylabel("Pseudobulk log2FC")
ax1.set_title("B. target=KLF1, compare L2FC for each DEG, Wilcoxon vs pseudobulk")
ax1.legend(fontsize=8)

# Panel C: pseudobulk LFC vs baseMean across all targets -- shows the
# low-count/high-LFC noise pattern is general, not specific to one target
ax2 = axes[2]
pb_plot = pb_all[pb_all["baseMean"] > 0]  # log-scale x needs positive values
ax2.scatter(pb_plot.loc[~pb_plot["significant"], "baseMean"],
            pb_plot.loc[~pb_plot["significant"], "log2FoldChange"],
            s=3, alpha=0.15, color="lightgrey", label="not significant")
ax2.scatter(pb_plot.loc[pb_plot["significant"], "baseMean"],
            pb_plot.loc[pb_plot["significant"], "log2FoldChange"],
            s=3, alpha=0.25, color="#55A868", label="significant (padj<0.05)")
ax2.set_xscale("log")
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_xlabel("DESeq2 baseMean: log(mean normalized count)")
ax2.set_ylabel("DESeq2 L2FC")
ax2.set_title("C. psuedobulk: low-count genes show inflated, noisy LFC")
ax2.legend(fontsize=8, markerscale=3)

fig.suptitle("Comparing single-cell Wilcoxon vs. pseudobulk DESeq2 for downstream DE", fontsize=13)
fig.tight_layout()
fig.savefig("../results/figures/de_method_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

Figure 3. Comparing single-cell (Wilcoxon) and pseudobulk (DESeq2) approaches to downstream perturbation DE.

(A) Jaccard overlap between each method's DEGs (padj < 0.05), computed per target (perturbation). 93 targets with sufficient data in both methods. Median overlap ≈0.31 — moderate agreement/dissonance (thousands of individual cells in Wilcoxon vs. 8 lanes ("replicates") in DESeq2).

(B) Gene-by-gene L2FC concordance for target/perturbation=KLF1. Red points are significant (padj < 0.05) in both Wilcoxon and pseudobulk DESeq2.

(C) DESeq2 pseudobulk: L2FC vs. mean normalized expression (baseMean, log scale) across all ~1.45M gene-target tests. Confirms that across all perturbations, low-expression genes show highly unstable, often extreme L2FC estimates regardless of padj, while higher-expression genes show progressively tighter, more reliable effect-size estimates.  lowly-detected genes have inflated L2FC, in positive direction - maybe because CRISPRa is likely to "occasionally induce gene count detection" for some genes with close to 0 counts in control. CRISPRi might resulted the inverted phenomenon: low basemean correlates with negative L2FC

In [ ]:
# code below shows what panel C does: that low-expressed genes usually have positive (not negative) inflated L2FC

low_expr_sig = pb_all[(pb_all["baseMean"] < 5) & (pb_all["significant"])]
print(low_expr_sig.shape)
print((low_expr_sig["log2FoldChange"] > 0).mean(), "fraction positive")
print(low_expr_sig.sort_values("log2FoldChange").head(5))   # most negative examples, if any exist
print(low_expr_sig.sort_values("log2FoldChange", ascending=False).head(5))  # most positive

# most negative examples have worse padj than most positive examples.
# just recapitulating figure panel C

In [ ]:
# ── Fold-activation vs. number of DE genes (paper found R≈0.07, i.e. essentially uncorrelated) ──

# 1. On-target log2FC: pull each target's own gene row from the Wilcoxon results
on_target = wilcoxon_de[wilcoxon_de["names"] == wilcoxon_de["target"]][
    ["target", "logfoldchanges"]
].rename(columns={"logfoldchanges": "on_target_log2fc"})

# 2. Number of significant DE genes per target (excluding the target gene itself)
de_counts = (
    wilcoxon_de[(wilcoxon_de["pvals_adj"] < 0.05) & (wilcoxon_de["names"] != wilcoxon_de["target"])]
    .groupby("target")
    .size()
    .rename("n_de_genes")
    .reset_index()
)

# 3. Merge — inner join drops any target missing an on-target row (shouldn't happen, but safe)
fold_vs_de = on_target.merge(de_counts, on="target", how="left")
fold_vs_de["n_de_genes"] = fold_vs_de["n_de_genes"].fillna(0).astype(int)

# 4. Correlation, matching the paper's reported statistic
from scipy.stats import pearsonr
r, p = pearsonr(fold_vs_de["on_target_log2fc"], fold_vs_de["n_de_genes"])
print(f"n={len(fold_vs_de)}, Pearson R={r:.2f}, p={p:.2e}")

In [ ]:
from scipy.stats import theilslopes

slope, intercept, lo, hi = theilslopes(
    fold_vs_de["n_de_genes"], fold_vs_de["on_target_log2fc"]
)

fig, ax = plt.subplots(figsize=(6, 5))

ax.scatter(
    fold_vs_de["on_target_log2fc"],
    fold_vs_de["n_de_genes"],
    s=25, alpha=0.6, color="#1f3d7a", edgecolor="white", linewidth=0.3,
)

x_line = np.array([fold_vs_de["on_target_log2fc"].min(), fold_vs_de["on_target_log2fc"].max()])
ax.plot(x_line, slope * x_line + intercept, linestyle="--", linewidth=1, color="gray", alpha=0.7)

ax.set_xlabel("On-target L2FC (activation strength)")
ax.set_ylabel("Number of significant DE genes\n(Wilcoxon, padj < 0.05)")
ax.set_title(f"CRISPRa target L2FC upreg vs. downstream DEG count\n(Pearson R = {r:.2f}, Theil-Sen slope = {slope:.2f}, n = {len(fold_vs_de)})")

fig.tight_layout()
fig.savefig("../results/figures/fold_activation_vs_de_genes.png", dpi=200)
plt.show()